In [1]:
!pip install torch torchvision timm

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm

In [3]:
DATA_DIR = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"
BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-4
NUM_CLASSES = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "Training"),
    transform=train_transform
)


In [7]:
val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "Testing"),
    transform=val_transform
)


In [8]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [9]:
model = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=True
)

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [10]:
in_features = model.head.in_features
model.head = nn.Linear(in_features, NUM_CLASSES)

In [16]:
model = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=True,
    num_classes=NUM_CLASSES
)

In [17]:
model = model.to(DEVICE)

In [18]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [19]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    return total_loss / len(train_loader), acc

In [20]:
def validate():
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    return total_loss / len(val_loader), acc

In [21]:
for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch()
    val_loss, val_acc = validate()

    scheduler.step()

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%")

torch.save(model.state_dict(), "swin_classifier.pth")
print("Model saved!")

100%|██████████| 50/50 [00:20<00:00,  2.50it/s]



Epoch [1/10]
Train Loss: 0.3381 | Train Acc: 87.68%
Val   Loss: 0.3754 | Val   Acc: 90.00%


100%|██████████| 50/50 [00:13<00:00,  3.60it/s]



Epoch [2/10]
Train Loss: 0.1465 | Train Acc: 95.38%
Val   Loss: 0.2965 | Val   Acc: 91.31%


100%|██████████| 50/50 [00:13<00:00,  3.66it/s]


...


Epoch [4/10]
Train Loss: 0.0672 | Train Acc: 97.98%
Val   Loss: 0.2822 | Val   Acc: 94.44%


100%|██████████| 50/50 [00:13<00:00,  3.67it/s]



Epoch [5/10]
Train Loss: 0.0585 | Train Acc: 98.21%
Val   Loss: 0.3324 | Val   Acc: 94.56%


 22%|██▏       | 38/175 [00:22<01:22,  1.66it/s]


KeyboardInterrupt: 